# Local PyMuPDF Extraction (Polling)

Companion to `local_pdf_pipeline.ipynb`. Run this in a **separate VS Code
kernel** alongside the download notebook so PyMuPDF text extraction happens
concurrently with the download — text appears in Drive as soon as PDFs land.

## Usage
1. Open `local_pdf_pipeline.ipynb` in one tab, run cells 3 + 5 (setup + download).
2. Open this notebook in another tab. VS Code gives it its own kernel.
3. Run cell 3 (setup) here — same Drive paths.
4. Run cell 5 (polling loop) here — every 60s it scans `pdfs/` and extracts
   any new ones that don't already have a `.txt` in `extracted/`.
5. Stop with the interrupt button when the download finishes (or leave it idle
   — empty passes cost nothing).

Both notebooks point at the **same** Drive folder, so anything you've already
extracted gets skipped automatically. Safe to run, stop, restart any time.

## Setup

Same configuration as the pipeline notebook — edit `DRIVE_PATH` if your Google
account email is different from the default.

In [ ]:
import sys
from pathlib import Path

# ============================================================
# CONFIGURATION (must match local_pdf_pipeline.ipynb)
# ============================================================
DRIVE_PATH = Path(
    "/Users/ernesto/Library/CloudStorage/GoogleDrive-ernestod1998@gmail.com"
    "/My Drive/litigation-pipeline"
)
REPO_DIR = Path("/Users/ernesto/Desktop/MLOPS-Project/litigation-outcome-pipeline")
# ============================================================

if not DRIVE_PATH.exists():
    raise RuntimeError(f"Drive Desktop sync not found at {DRIVE_PATH}")
if not REPO_DIR.exists():
    raise RuntimeError(f"Repo not found at {REPO_DIR}")

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

PDF_DIR = DRIVE_PATH / "pdfs"
OUTPUT_DIR = DRIVE_PATH / "extracted"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"PDF dir:       {PDF_DIR}")
print(f"Output dir:    {OUTPUT_DIR}")
print(f"PDFs in Drive: {len(list(PDF_DIR.glob('*.pdf')))}")
print(f".txt files:    {len(list(OUTPUT_DIR.glob('*.txt')))}")

## Polling extraction loop

Every 60 seconds:
- list whitelisted PDFs in `PDF_DIR`
- skip any with an existing `.txt` in `OUTPUT_DIR`
- run PyMuPDF on the rest
- write `.txt` if extracted text > 100 chars; queue PDF for GPU otherwise
- log corrupt PDFs (partial downloads / HTML error pages) and skip

The loop runs forever — interrupt the cell when you're done.

In [ ]:
import time

import fitz  # PyMuPDF

from scraper.config import is_doc_type_wanted as is_doc_wanted

MIN_TEXT_LENGTH = 100
POLL_INTERVAL = 60  # seconds between passes


def extract_with_pymupdf(pdf_path):
    doc = fitz.open(str(pdf_path))
    pages = [page.get_text() for page in doc]
    doc.close()
    return "\n\n".join(pages).strip()


pass_num = 0
cumulative_extracted = 0
cumulative_needs_gpu = 0
cumulative_corrupt = 0

while True:
    pass_num += 1
    extracted_this_pass = 0
    needs_gpu_this_pass = 0
    corrupt_this_pass = 0

    all_pdfs = sorted(PDF_DIR.glob("*.pdf"))
    pdfs = [
        p
        for p in all_pdfs
        if is_doc_wanted(p.stem.split("_", 1)[-1] if "_" in p.stem else p.stem)
    ]

    for pdf_path in pdfs:
        txt_path = OUTPUT_DIR / f"{pdf_path.stem}.txt"
        if txt_path.exists():
            continue

        try:
            text = extract_with_pymupdf(pdf_path)
        except Exception as e:
            size = pdf_path.stat().st_size if pdf_path.exists() else 0
            print(
                f"  CORRUPT ({size} bytes): {pdf_path.name} \u2014 {type(e).__name__}: {e}",
                flush=True,
            )
            corrupt_this_pass += 1
            continue

        if len(text) > MIN_TEXT_LENGTH:
            txt_path.write_text(text, encoding="utf-8")
            extracted_this_pass += 1
        else:
            needs_gpu_this_pass += 1

    cumulative_extracted += extracted_this_pass
    cumulative_needs_gpu += needs_gpu_this_pass
    cumulative_corrupt += corrupt_this_pass

    print(
        f"pass {pass_num}: +{extracted_this_pass} extracted, "
        f"{needs_gpu_this_pass} need GPU, {corrupt_this_pass} corrupt "
        f"(cumulative: {cumulative_extracted} / {cumulative_needs_gpu} / {cumulative_corrupt}) "
        f"\u2014 sleeping {POLL_INTERVAL}s",
        flush=True,
    )
    time.sleep(POLL_INTERVAL)